# Data loading: URL, local file, and numpy array

Three ways to get data into ipyniivue:

1. **From a URL** \u2014 `nv.add_volume_from_url(url)` (Phase A path).
2. **From a local file** \u2014 `nv.add_volume_from_path(path)` reads the file in Python and ships the bytes to the browser. The browser does not need filesystem access.
3. **From a numpy array** \u2014 `nv.add_volume_from_array(arr, affine=...)` builds a NIfTI in memory via [nibabel](https://nipy.org/nibabel/) and ships the bytes.

Mesh equivalents: `add_mesh_from_url`, `add_mesh_from_path`, `add_mesh_from_bytes`.

Install the optional deps for this notebook with `pip install ipyniivue[examples]` (numpy + nibabel).


In [ ]:
import pathlib

import numpy as np
import nibabel as nib
from IPython.display import display
from ipyniivue import NiiVue


## 1. Load from URL

The simplest path. The browser fetches the file directly.

In [ ]:
nv1 = NiiVue(slice_type=3, is_colorbar_visible=True, backend="webgl2")
display(nv1)

nv1.add_volume_from_url(
    "https://niivue.github.io/mono/volumes/mni152.nii.gz",
    cal_min=30, cal_max=80, colormap="gray",
)


## 2. Load from local path

`fslmean8.nii.gz` (ships next to this notebook, 38 KB, 64x64x36 voxels at ~3 mm) loads from disk through the same browser canvas. The Python kernel reads the file; the bytes ride the synced `_msg_inbox` trait as base64 (the `_b64` field on the command body) and JS decodes them back to a `Uint8Array` before handing them to NiiVue's URL/File loader, which dispatches by filename extension. The 33% base64 overhead is the cost of one cold-start-safe channel; the inbox is pruned via `_msg_inbox_ack` after JS drains so payloads don't pin trait state across the session.

In [ ]:
nv2 = NiiVue(slice_type=3, is_colorbar_visible=True, backend="webgl2")
display(nv2)

local_path = pathlib.Path("fslmean8.nii.gz")
nv2.add_volume_from_path(local_path, colormap="gray")


## 3. Load from a numpy array

Build a 64\u00b3 sphere of intensity falling off from the center, wrap it in a NIfTI-1 with an identity affine, and display it. `add_volume_from_array` calls nibabel internally to construct the header and serialize.

In [ ]:
nv3 = NiiVue(slice_type=3, is_colorbar_visible=True, backend="webgl2")
display(nv3)

size = 64
ix, iy, iz = np.indices((size, size, size))
center = np.array([size // 2] * 3)
dist = np.sqrt((ix - center[0]) ** 2 + (iy - center[1]) ** 2 + (iz - center[2]) ** 2)
sphere = np.clip(1.0 - dist / 20.0, 0.0, 1.0).astype(np.float32)

nv3.add_volume_from_array(
    sphere,
    affine=np.eye(4),
    name="sphere.nii",
    colormap="hot",
)


## 4. Read with nibabel, modify, then load

Read the local volume with nibabel, add a vertical rod at the center, and ship the modified bytes. Nothing touches disk after the initial read.

In [ ]:
nv4 = NiiVue(slice_type=3, is_colorbar_visible=True, backend="webgl2")
display(nv4)

img = nib.load("fslmean8.nii.gz")
data = img.get_fdata().astype(np.float32)
max_val = data.max()

x_center, y_center = data.shape[0] // 2, data.shape[1] // 2
for z in range(2, data.shape[2] - 2):
    for dx in range(-2, 3):
        for dy in range(-2, 3):
            if dx * dx + dy * dy <= 4:
                data[x_center + dx, y_center + dy, z] = max_val * 2

nv4.add_volume_from_array(
    data,
    affine=img.affine,
    name="modified.nii",
    colormap="gray",
)
